# Recurrences and Recursive Cost


```{contents}
:local:
:depth: 2
```

A recurrence models work by input size. Keep the returned value, total calls, and maximum simultaneously active frames separate. These examples preview recursive analysis; Chapter 21 develops recursive programming.

## Base Cases and Linear Recurrences

```{index} Base Cases and Linear Recurrences
```


Let C(n) count calls, including the base call. For a nonnegative countdown, C(0)=1 and C(n)=C(n−1)+1. Expanding gives C(n)=n+1. The returned value below is n, which is a different quantity.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,4 })
{
    int calls=0, active=0, peak=0;
    int result=Count(n);
    Console.WriteLine($"n={n}, result={result}, calls={calls}, peak frames={peak}");
    int Count(int size)
    {
        calls++; active++; peak=Math.Max(peak,active);
        int value = size == 0 ? 0 : 1 + Count(size-1);
        active--; return value;
    }
}

Time and stack space are Θ(n+1); for growing positive n, write Θ(n). Each call has constant local storage. Negative input would never reach zero in this model, so nonnegative input is a required precondition. Production recursive code needs a practical depth limit; .NET does not promise unlimited recursion or tail-call elimination.

## Halving with Integer Inputs

```{index} Halving with Integer Inputs
```


C(n)=1 for n≤1 and C(n)=C(floor(n/2))+1 otherwise. For n≥1 this gives floor(log₂ n)+1 calls. The number of reductions is one less than the number of calls.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,3,8,9 })
    Console.WriteLine($"n={n}, calls={HalvingCalls(n)}");
int HalvingCalls(int size)
{
    if (size < 0) throw new ArgumentOutOfRangeException(nameof(size));
    return size <= 1 ? 1 : 1 + HalvingCalls(size/2);
}

For powers of two, the chain is n, n/2, …, 1. For n=9 the integer chain is 9,4,2,1. Depth and time are Θ(log n) for large n; zero and one remain constant-cost base cases.

## Two Branches and Work per Level

```{index} Two Branches and Work per Level
```


For a model with n=2^k leaves, define W(1)=1 and W(n)=2W(n/2)+n. At every internal level, the total combine work is n. There are log₂ n internal levels, plus n leaf units, so W(n)=n log₂ n+n. This models a merge-style algorithm without implementing sorting here.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 1,2,4,8,16 })
{
    long calls=0; int active=0, peak=0;
    long work=Work(n);
    Console.WriteLine($"n={n}, work={work}, calls={calls}, peak frames={peak}");
    long Work(int size)
    {
        calls++; active++; peak=Math.Max(peak,active);
        long result = size == 1 ? 1 : Work(size/2)+Work(size/2)+size;
        active--;return result;
    }
}

For n=8, level work is 8+8+8+8=32, total calls are 15, and peak active frames are 4. Sibling calls occur sequentially, so they increase total calls without doubling stack depth. This counter program itself visits Θ(n) nodes; its returned work models an algorithm that actually performs n combine operations at each internal node.

## Rounding and Auxiliary Storage

```{index} Rounding and Auxiliary Storage
```


For arbitrary positive sizes, split into floor(n/2) and ceil(n/2), preserving all items. Using floor(n/2) twice loses an item for odd n. For n>1, both parts must be smaller than n.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 1,3,5,8,9 })
    Console.WriteLine($"n={n}, modeled work={BalancedWork(n)}");
long BalancedWork(int n)
{
    if(n < 1) throw new ArgumentOutOfRangeException(nameof(n));
    if(n == 1) return 1;
    int left=n/2, right=n-left;
    return BalancedWork(left)+BalancedWork(right)+n;
}

The general model remains Θ(n log n) work. Its stack depth is Θ(log n), with constant storage per frame. A real merge sort usually needs Θ(n) auxiliary merge storage as well. Stack depth alone is not total auxiliary space. State how buffers are allocated and reused before claiming a memory bound.

## Branching and Repeated Subproblems

```{index} Branching and Repeated Subproblems
```


The naive Fibonacci call count obeys C(0)=C(1)=1 and C(n)=C(n−1)+C(n−2)+1. Repeated subproblems create exponential work. A recursion tree can be wide even when its depth is linear.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 0,1,4,6,8 })
{
    long calls=0;int active=0,peak=0;
    int value=Fib(n);
    Console.WriteLine($"n={n}, Fib={value}, calls={calls}, peak frames={peak}");
    int Fib(int size)
    {
        calls++;active++;peak=Math.Max(peak,active);
        int result=size<=1?size:Fib(size-1)+Fib(size-2);
        active--;return result;
    }
}

For n=6 there are 25 calls but only 6 active frames at the peak. The time is O(2^n), a useful upper bound; the tighter exponential growth is Θ(φ^n), where φ≈1.618. Stack space is Θ(n). Memoization reduces repeated work by retaining results; its implementation belongs in Chapter 24. Derive a recurrence from calls and nonrecursive work, include its base case, and distinguish the model from the code that calculates an estimate.

## Exercise

```{index} Analysis practice
```

For n=8, print the number of leaves and total calls in a balanced two-branch recursion with constant work per node. Contrast this with a linear combine-cost model.

In [ ]:
using System;

// Exercise
// For n=8, print the number of leaves and total calls in a balanced two-branch recursion with constant work per node. Contrast this with a linear combine-cost model.
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;

// Solution
int calls=0;
Console.WriteLine($"leaves={Leaves(8)}, calls={calls}");
int Leaves(int n)
{
 calls++;
 return n==1?1:Leaves(n/2)+Leaves(n/2);
}
// C(1)=1, C(n)=2C(n/2)+1; Theta(n) calls, Theta(log n) stack.
// A linear combine cost changes modeled work to Theta(n log n).

leaves=8, calls=15


```{rubric} Footnotes
```
[^1]: Recursion depth here counts active calls including the base call. Some texts count edges between calls instead; specify the convention.
[^2]: The power-of-two assumption simplifies exact level sums. Balanced integer splits retain the same asymptotic bound for other positive sizes.